In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as tt

from sklearn.datasets import make_regression

In [ ]:
data = pd.read_csv('../Data/res_demo_1.csv').sample(20)

In [ ]:
data.head()

In [ ]:
plt.scatter(data.x, data.y_fan_back)

In [ ]:
N = data.shape[0]

with pm.Model() as simple_linear:

    intercept = pm.Normal("intercept", mu=0, sigma=1, shape=1)
    betas = pm.Normal("betas", mu=0, sigma=1, shape=1)

    regression = pm.Deterministic("regression", intercept + betas*data.x)
    y = pm.Normal("y", mu=regression, sigma=100, observed=data.y_fan_back)

In [ ]:
with simple_linear:
    trace_lin = pm.sample(5000, tune=200, chains=2, target_accept=0.90, return_inferencedata=True)

    # check convergence diagnostics
    assert all(az.rhat(trace_lin) < 1.03)

In [ ]:
az.plot_trace(trace_lin, var_names=["intercept","betas","regression"])
plt.show()

In [ ]:
pd.DataFrame({
    'x_': np.tile(data.x,10000),
    'regression': trace_lin.posterior.regression.to_numpy().reshape(-1)
}).groupby('x_').regression.agg([
    ('upper',lambda x: x.quantile(0.975)),
    ('mean', lambda x: x.mean()),
    ('lower',lambda x: x.quantile(0.025))]).plot()